# Complete FastHTML Application Example

This notebook demonstrates a more complete application with multiple endpoints, error handling, and job status tracking.

In [1]:
from fasthtml.common import *
from starlette.responses import StreamingResponse, JSONResponse
import uuid, time, json

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream
from cjm_tqdm_capture.progress_info import serialize_all_jobs

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app with history tracking enabled
app, rt = create_test_app(theme=DaisyUITheme.DARK)

# Initialize with history for debugging
monitor = ProgressMonitor(keep_history=True, history_limit=100)
runner = JobRunner(monitor)

# Store job results
job_results = {}

In [4]:
# Define a complex task with parameters and multiple stages
def complex_task(task_name, iterations, speed="normal"):
    from tqdm import tqdm
    import time
    import random
    
    speeds = {"fast": 0.001, "normal": 0.01, "slow": 0.05}
    delay = speeds.get(speed, 0.01)
    
    results = {"task": task_name, "stages": []}
    
    # Stage 1: Initialization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Initialize"):
        time.sleep(delay)
    results["stages"].append("Initialization complete")
    
    # Stage 2: Processing
    processed_data = []
    for i in tqdm(range(iterations), desc=f"{task_name}: Process"):
        time.sleep(delay)
        processed_data.append(i * random.random())
    results["stages"].append(f"Processed {len(processed_data)} items")
    
    # Stage 3: Validation
    for i in tqdm(range(iterations // 2), desc=f"{task_name}: Validate"):
        time.sleep(delay)
    results["stages"].append("Validation complete")
    
    # Stage 4: Finalization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Finalize"):
        time.sleep(delay)
    results["stages"].append("Finalization complete")
    
    results["summary"] = f"Task {task_name} completed successfully with {iterations} iterations"
    return results

In [5]:
# Enhanced UI with form inputs
@rt("/")
def home():
    return create_test_page(
        "Complete Progress Application",
        Div(
            H1("Task Manager with Progress Tracking", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Task configuration form
            Div(
                H2("Configure Task", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Task Name:", cls=str(label)),
                    Input(id="task-name", type="text", value="MyTask", cls=combine_classes(text_input, w.full, max_w.xs)),
                    cls=combine_classes(m.b(4))
                ),
                Div(
                    Label("Iterations:", cls=str(label)),
                    Input(id="iterations", type="number", value="100", min="10", max="500", cls=combine_classes(text_input, w.full, max_w.xs)),
                    cls=combine_classes(m.b(4))
                ),
                Div(
                    Label("Speed:", cls=str(label)),
                    Select(
                        Option("Fast", value="fast"),
                        Option("Normal", value="normal", selected=True),
                        Option("Slow", value="slow"),
                        id="speed",
                        cls=combine_classes(select, w.full, max_w.xs)
                    ),
                    cls=combine_classes(m.b(4))
                ),
                Button("Start Task", id="start-btn", cls=combine_classes(btn, btn_colors.primary)),
                Button("Clear Completed", id="clear-btn", cls=combine_classes(btn, btn_colors.secondary, m.l(2))),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress display
            Div(
                H2("Current Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    P("Overall:", cls=str(font_weight.bold)),
                    Progress(id="overall-progress", max="100", value="0", cls=combine_classes(progress, progress_colors.primary, w.full, m.b(2))),
                    P("0%", id="overall-text", cls=combine_classes(font_size.sm, m.b(4))),
                    
                    Div(id="progress-bars", cls=str(space.y(2))),
                    
                    Pre("Ready", id="status", cls=combine_classes(bg_dui.base_300, p(4), rounded(), m.t(4), font_size.xs)),
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Active jobs list
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div("No active jobs", id="jobs-list", cls=str(overflow.x.auto)),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        ),
        Script("""
            let currentSSE = null;
            let currentJobId = null;
            
            const startBtn = document.getElementById('start-btn');
            const clearBtn = document.getElementById('clear-btn');
            const overallProgress = document.getElementById('overall-progress');
            const overallText = document.getElementById('overall-text');
            const progressBars = document.getElementById('progress-bars');
            const status = document.getElementById('status');
            const jobsList = document.getElementById('jobs-list');
            
            // Refresh jobs list
            async function refreshJobs() {
                const resp = await fetch('/api/jobs');
                const jobs = await resp.json();
                
                if (Object.keys(jobs).length === 0) {
                    jobsList.textContent = 'No active jobs';
                } else {
                    const table = document.createElement('table');
                    table.className = '""" + combine_classes(table, table_modifiers.zebra, w.full) + """';
                    
                    const thead = document.createElement('thead');
                    thead.innerHTML = '<tr><th>Job ID</th><th>Progress</th><th>Status</th><th>Action</th></tr>';
                    table.appendChild(thead);
                    
                    const tbody = document.createElement('tbody');
                    for (const [jobId, job] of Object.entries(jobs)) {
                        const row = document.createElement('tr');
                        row.innerHTML = `
                            <td>${jobId.substring(0, 8)}...</td>
                            <td>${job.overall_progress.toFixed(1)}%</td>
                            <td>${job.completed ? 'Complete' : 'Running'}</td>
                            <td><button class=\"""" + combine_classes(btn, btn_sizes.xs) + """\" onclick="viewJob('${jobId}')">View</button></td>
                        `;
                        tbody.appendChild(row);
                    }
                    table.appendChild(tbody);
                    
                    jobsList.innerHTML = '';
                    jobsList.appendChild(table);
                }
            }
            
            // View specific job
            window.viewJob = async (jobId) => {
                if (currentSSE) {
                    currentSSE.close();
                }
                
                currentJobId = jobId;
                status.textContent = `Viewing job: ${jobId}`;
                
                // Get current status
                const resp = await fetch(`/api/jobs/${jobId}`);
                const data = await resp.json();
                
                if (data.error) {
                    status.textContent = `Job not found: ${jobId}`;
                    return;
                }
                
                // Connect to SSE for this job
                connectSSE(jobId);
            };
            
            // Connect SSE
            function connectSSE(jobId) {
                if (currentSSE) {
                    currentSSE.close();
                }
                
                currentSSE = new EventSource(`/api/jobs/${jobId}/stream`);
                
                currentSSE.onmessage = (e) => {
                    const data = JSON.parse(e.data);
                    
                    // Update overall progress
                    overallProgress.value = data.progress || 0;
                    overallText.textContent = `${(data.progress || 0).toFixed(1)}%`;
                    
                    // Update individual progress bars
                    progressBars.innerHTML = '';
                    if (data.bars) {
                        for (const [barId, bar] of Object.entries(data.bars)) {
                            const barDiv = document.createElement('div');
                            barDiv.className = '""" + str(m.b(2)) + """';
                            barDiv.innerHTML = `
                                <p class=\"""" + combine_classes(font_size.sm, font_weight.semibold) + """\">${bar.desc || 'Progress'}</p>
                                <progress class=\"""" + combine_classes(progress, progress_colors.accent, w.full) + """\" value="${bar.pct}" max="100"></progress>
                                <p class=\"""" + str(font_size.xs) + """\">${bar.pct.toFixed(1)}% (${bar.cur}/${bar.tot})</p>
                            `;
                            progressBars.appendChild(barDiv);
                        }
                    }
                    
                    // Update status
                    if (data.completed) {
                        status.textContent = `Job ${jobId} completed!`;
                        startBtn.disabled = false;
                        currentSSE.close();
                        currentSSE = null;
                        refreshJobs();
                        
                        // Fetch and display results
                        fetchResults(jobId);
                    }
                };
                
                currentSSE.onerror = (err) => {
                    console.error('SSE error:', err);
                    // Don't close on all errors - SSE will auto-reconnect
                    status.textContent = 'Connection interrupted...';
                    startBtn.disabled = false;
                    refreshJobs();
                };
            }
            
            // Fetch job results
            async function fetchResults(jobId) {
                const resp = await fetch(`/api/jobs/${jobId}/result`);
                if (resp.ok) {
                    const result = await resp.json();
                    status.textContent = `Job completed! Result: ${JSON.stringify(result, null, 2)}`;
                }
            }
            
            // Start button handler
            startBtn.onclick = async () => {
                startBtn.disabled = true;
                
                // Close existing SSE if any
                if (currentSSE) {
                    currentSSE.close();
                    currentSSE = null;
                }
                
                // Get form values
                const taskName = document.getElementById('task-name').value;
                const iterations = parseInt(document.getElementById('iterations').value);
                const speed = document.getElementById('speed').value;
                
                // Create job
                const resp = await fetch('/api/jobs', {
                    method: 'POST',
                    headers: {'Content-Type': 'application/json'},
                    body: JSON.stringify({task_name: taskName, iterations, speed})
                });
                const data = await resp.json();
                currentJobId = data.job_id;
                
                status.textContent = `Started job: ${currentJobId}`;
                
                // Connect SSE
                connectSSE(currentJobId);
                
                // Refresh jobs list
                refreshJobs();
            };
            
            // Clear completed jobs
            clearBtn.onclick = async () => {
                const resp = await fetch('/api/jobs/clear', {method: 'POST'});
                if (resp.ok) {
                    refreshJobs();
                    status.textContent = 'Cleared completed jobs';
                } else {
                    status.textContent = 'Failed to clear jobs';
                }
            };
            
            // Initial refresh
            refreshJobs();
            setInterval(refreshJobs, 5000); // Auto-refresh every 5 seconds
        """)
    )

In [6]:
# API endpoints
@rt("/api/jobs", methods=["POST"])
async def create_job(request):
    data = await request.json()
    job_id = str(uuid.uuid4())
    
    # Wrapper to store results
    def task_wrapper():
        result = complex_task(
            data.get('task_name', 'Task'),
            data.get('iterations', 100),
            data.get('speed', 'normal')
        )
        job_results[job_id] = result
    
    # Adjust throttling based on speed to prevent apparent freezing
    speed = data.get('speed', 'normal')
    patch_config = {
        'fast': {"min_delta_pct": 10, "min_update_interval": 0.01, "emit_initial": True},
        'normal': {"min_delta_pct": 5, "min_update_interval": 0.05, "emit_initial": True},
        'slow': {"min_delta_pct": 2, "min_update_interval": 0.2, "emit_initial": True}
    }
    
    runner.start(
        job_id,
        task_wrapper,
        patch_kwargs=patch_config.get(speed, patch_config['normal'])
    )
    
    return JSONResponse({"job_id": job_id})

@rt("/api/jobs", methods=["GET"])
def list_jobs():
    jobs = monitor.all()
    # Use the helper function to serialize
    return JSONResponse(serialize_all_jobs(jobs))

@rt("/api/jobs/{job_id}")
def get_job(job_id: str):
    snapshot = monitor.snapshot(job_id)
    if not snapshot:
        return JSONResponse({"error": "Job not found"}, status_code=404)
    
    return JSONResponse({
        "job_id": job_id,
        "progress": snapshot["overall_progress"],
        "completed": snapshot["completed"],
        "bars": {
            k: {
                "description": v.description,
                "progress": v.progress,
                "current": v.current,
                "total": v.total
            }
            for k, v in snapshot["bars"].items()
        }
    })

@rt("/api/jobs/{job_id}/result")
def get_job_result(job_id: str):
    if job_id in job_results:
        return JSONResponse(job_results[job_id])
    return JSONResponse({"error": "No results yet"}, status_code=404)

@rt("/api/jobs/{job_id}/stream")
def stream_job(job_id: str):
    gen = sse_stream(
        monitor,
        job_id,
        interval=0.1,
        heartbeat=10.0,
        wait_for_start=True,
        start_timeout=30.0
    )
    return StreamingResponse(gen, media_type="text/event-stream")

@rt("/api/jobs/clear", methods=["POST"])
def clear_completed():
    monitor.clear_completed(older_than_seconds=0)
    # Also clear results for completed jobs
    for job_id in list(job_results.keys()):
        if not monitor.snapshot(job_id):
            del job_results[job_id]
    return JSONResponse({"status": "cleared"})

In [9]:
# Start server
server = start_test_server(app)
display(HTMX())

MyTask: Finalize: 100%|███████████████████████████████████████████████████| 25/25 [00:01<00:00, 19.87it/s]


In [10]:
# Stop server when done
server.stop()